<a href="https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moath177/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content item (`content_hash_id`), aggregated over a single month window.

**Tables:** `dim_content` (content metadata) joined to `fact_content_daily_performance` (daily search performance, filtered to one month partition).

**Time window:** development and verification are done on `month = 2026-03` — a mid-panel month, not the final month. The `_sample` table (June 2026) is explicitly avoided for anything beyond mechanics testing, since it is the natural outcome window for any past→future label and would leak into later modeling weeks.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token (hf_...): ··········


In [13]:
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    HAVING n_clients > 1
    LIMIT 5
""").df()
print(f'content items spanning >1 client: {len(grain_check)}')

window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(*) AS n_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(window_check['min_date'][0], window_check['max_date'][0], window_check['n_rows'][0])

content items spanning >1 client: 0
2026-03-01 00:00:00 2026-03-31 00:00:00 9841378


**Verified:** zero content items appeared under more than one `client_hash_id` in 2026-03 (grain holds). `report_date` spans the full month (2026-03-01 to 2026-03-31), 9,841,378 rows total before any visibility filter.

## 2. Fields: feature / label / context / excluded

**Field classification for Lane 4 (CTR/Engagement Opportunity Scoring), month = 2026-03:**

### Context
*(grouping, joining, or filtering — never fed to the model)*
- `content_hash_id`, `client_hash_id` — join/group keys only (verified: no content item spans multiple clients in this month)
- `report_date` / `month` — time window key
- `is_published`, `is_deleted` — used only to define the analysis slice: **visible = `is_published IS TRUE AND is_deleted IS FALSE`**. Not passed to the model.

### Label / proxy
- `ctr = gsc_clicks / gsc_impressions` — raw click-through rate for this first warehouse pass. Not position/tier-adjusted yet (opportunity_gap deferred to a later iteration).

### Feature
*(knowable before the decision moment, safe to use)*
- `gsc_avg_position`
- `gsc_impressions`
- `content_type`
- `word_count`
- content age (derived: `report_date - content_created_date`)

### Excluded
- `gsc_clicks` — used only to compute the label; excluded as a feature to avoid direct leakage (the label is literally built from it).
- GA4/engagement columns (e.g. sessions, engagement_rate) — not used in this pass. Would require filtering on `ga4_data_available IS TRUE` first, since rows before a client's GA4 start are zero-filled, not truly zero-engagement. Deferred, not forgotten.

gsc_clicks — used only to compute the label; excluded as a feature to avoid direct leakage (the label is literally built from it).
GA4/engagement columns (e.g. sessions, engagement_rate) — not used in this pass. Would require filtering on ga4_data_available IS TRUE first, since rows before a client's GA4 start are zero-filled, not truly zero-engagement. Deferred, not forgotten.

In [4]:
schema_check = con.sql(f"""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet'))
    WHERE column_name IN ('content_hash_id','content_type','word_count','content_created_date','is_published','is_deleted')

    UNION ALL

    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet'))
    WHERE column_name IN ('client_hash_id','content_hash_id','report_date','gsc_avg_position','gsc_impressions','gsc_clicks')
""").df()

schema_check

,column_name,column_type
0,content_hash_id,VARCHAR
1,content_created_date,DATE
2,content_type,VARCHAR
3,word_count,BIGINT
4,is_published,BOOLEAN
5,is_deleted,BOOLEAN
6,report_date,DATE
7,client_hash_id,VARCHAR
8,content_hash_id,VARCHAR
9,gsc_impressions,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on month = 2026-03, followed by the five-feature frame and a deliberate leakage demonstration.


In [14]:
grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    HAVING n_clients > 1
    LIMIT 5
""").df()

print(f'content items spanning >1 client: {len(grain_check)}')
grain_check

content items spanning >1 client: 0


,content_hash_id,n_clients


**Grain verified:** zero content items appeared under more than one `client_hash_id` in 2026-03 — one row = one content item holds without ambiguity.

In [15]:
total_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows_total
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows_visible,
        COUNT(DISTINCT f.content_hash_id) AS n_content_items,
        MIN(f.report_date) AS min_date,
        MAX(f.report_date) AS max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d
        ON f.content_hash_id = d.content_hash_id
    WHERE d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""").df()

print(total_check)
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_rows_total
0       9841378
   n_rows_visible  n_content_items   min_date   max_date
0         9532718           321106 2026-03-01 2026-03-31


**Counts, date span, and availability:** of 9,841,378 total rows in 2026-03, 9,532,718 rows (≈96.9%) belong to visible content (`is_published IS TRUE AND is_deleted IS FALSE`), spanning 321,106 distinct content items across the full month (2026-03-01 to 2026-03-31).

In [16]:
feature_frame = con.sql(f"""
    WITH visible_content AS (
        SELECT content_hash_id, content_type, word_count, content_created_date
        FROM read_parquet('{REL}/dim_content.parquet')
        WHERE is_published IS TRUE
          AND is_deleted IS FALSE
    ),
    monthly_agg AS (
        SELECT
            f.content_hash_id,
            AVG(f.gsc_avg_position)              AS avg_position,
            SUM(f.gsc_impressions)                AS impressions,
            SUM(f.gsc_clicks)                     AS clicks,
            MAX(f.report_date)                    AS last_report_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
        JOIN visible_content v
            ON f.content_hash_id = v.content_hash_id
        GROUP BY f.content_hash_id
    )
    SELECT
        m.content_hash_id,
        m.avg_position,
        m.impressions,
        m.clicks,
        CASE WHEN m.impressions > 0
             THEN m.clicks::DOUBLE / m.impressions
             ELSE NULL END                        AS ctr,
        v.content_type,
        v.word_count,
        DATE_DIFF('day', v.content_created_date, m.last_report_date) AS content_age_days
    FROM monthly_agg m
    JOIN visible_content v
        ON m.content_hash_id = v.content_hash_id
""").df()

print(f'{len(feature_frame):,} rows')
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

321,106 rows


,content_hash_id,avg_position,impressions,clicks,ctr,content_type,word_count,content_age_days
0,content_7a105f548d9c6916,7.209549,6523.0,7.0,0.001073,keyword article,2123,396
1,content_a3ea9792f793ec72,2.987198,453.0,0.0,0.000000,keyword article,<NA>,396
2,content_36c36abc7650d7af,6.724039,5630.0,6.0,0.001066,keyword article,2546,396
3,content_a7da352b73b02668,7.244844,4944.0,13.0,0.002629,keyword article,2330,396
4,content_f39be42b42a4e8f6,14.432540,42.0,0.0,0.000000,keyword article,<NA>,396


**Five-feature frame** (321,106 rows, one per visible content item, aggregated over 2026-03):

- `avg_position` — average of `gsc_avg_position` across the month. *Available when: every day search performance is measured — known before any editorial decision.*
- `impressions` — sum of `gsc_impressions` across the month. *Available when: accumulates daily, fully known by month end, before the decision moment.*
- `content_type` — from `dim_content`. *Available when: set at content creation, fixed before the month begins.*
- `word_count` — from `dim_content`. *Available when: set at publish/last edit, known before the month begins.*
- `content_age_days` — derived (`last_report_date - content_created_date`). *Available when: both inputs are known before the decision moment (creation date is historical, last report date is within the observed window).*

Note: `ctr` is the label/proxy (see section 2) — not a feature, never fed to the model.

### The leakage trap

A minimum-volume filter (`impressions >= 50`) is applied first to remove low-count noise. An honest quick score is built from features only, then deliberately contaminated with the label, to see how much the score inflates.

In [19]:
qf = feature_frame[feature_frame['impressions'] >= 50].copy()
print(f'{len(qf):,} rows after minimum-volume filter (was {len(feature_frame):,})')

qf['quick_score_honest'] = qf['impressions'] / qf['avg_position']

K = 25
true_top_k = set(qf.sort_values('ctr', ascending=False).head(K)['content_hash_id'])
predicted_top_k_honest = set(qf.sort_values('quick_score_honest', ascending=False).head(K)['content_hash_id'])

precision_at_k_honest = len(predicted_top_k_honest & true_top_k) / K
print(f'Honest Precision@{K}: {precision_at_k_honest:.3f}')

116,063 rows after minimum-volume filter (was 321,106)
Honest Precision@25: 0.000


In [20]:
# ⚠️ DELIBERATE LEAK — for demonstration only, never keep this
qf['quick_score_leaky'] = qf['quick_score_honest'] + qf['ctr'] * 1_000_000

predicted_top_k_leaky = set(qf.sort_values('quick_score_leaky', ascending=False).head(K)['content_hash_id'])
precision_at_k_leaky = len(predicted_top_k_leaky & true_top_k) / K
print(f'Leaky Precision@{K}: {precision_at_k_leaky:.3f}')

Leaky Precision@25: 0.760


In [21]:
qf = qf.drop(columns=['quick_score_leaky'])

print('Deliberate leak removed. Honest score kept.')
print(f'Honest Precision@{K}: {precision_at_k_honest:.3f}  ← the number we actually trust')

Deliberate leak removed. Honest score kept.
Honest Precision@25: 0.000  ← the number we actually trust


**Result:** the honest score (built only from `impressions` and `avg_position`) achieved Precision@25 = 0.000 against true top-CTR pages — notably low, likely because raw volume/position and raw CTR rank very different pages at this granularity. Deliberately adding the label (`ctr`) into the score pushed Precision@25 to 0.760 — a clear artificial jump confirming circular measurement, not genuine signal. The leaked column was removed; only the honest 0.000 score is kept as the real (if weak) baseline for this first pass.

In [17]:
missing_word_count = feature_frame['word_count'].isna().mean()
print(f'word_count missing in {missing_word_count:.1%} of rows')

word_count missing in 32.9% of rows


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Missingness in `word_count` follows `content_type`, not randomness:** 32.9% of visible content items are missing `word_count` overall, but this is highly uneven across types — `keyword article` (the most common type) is missing 37.5% of the time, versus 5.7% for `feedly article` and under 0.1% for `comparison article`.

This means the data can never distinguish, for a large share of `keyword article` pages, between "genuinely thin content" and "word count was never recorded." A blind `fillna(0)` would silently encode a content-type signal into the feature rather than a true zero. Any downstream use of `word_count` must add a `has_word_count` flag rather than filling blindly — this limitation is directional evidence for future feature work, not something this contract can resolve on its own.

In [23]:
missing_word_count = feature_frame['word_count'].isna().mean()
print(f'word_count missing in {missing_word_count:.1%} of rows overall')

missing_by_type = feature_frame.groupby('content_type')['word_count'].apply(lambda s: s.isna().mean())
missing_by_type

word_count missing in 32.9% of rows overall


,word_count
content_type,
comparison article,0.000885
feedly article,0.056989
keyword article,0.374906


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.